### eBird Data Exploration: Great-tailed Grackle (*Quiscalus mexicanus*)
---
**Capstone Project · DataTalks.Club 2026**  
**Author:** Victoria Torreno  
**Source:** [eBird API 2.0](https://documenter.getpostman.com/view/664302/S1ENwy59)  
**Citation:** Cornell Lab of Ornithology. eBird Basic Dataset. Version: EBD_relFeb-2026. Cornell Lab of Ornithology, Ithaca, New York, 2026.  
**Goal:** Verify API connectivity, inspect raw JSON, and define the Bronze layer schema.  

#### Setup

In [ ]:
import os
import pandas as pd
from ebird.api.requests import get_species_observations
from dotenv import load_dotenv

In [ ]:
load_dotenv()
api_key = os.getenv('EBIRD_API_KEY')

In [ ]:
species_code = 'grtgra' # great-tailed grackle
target_regions = ['US-AZ', 'US-TX', 'US-CA', 'US-NM', 'US-OK']

#### API Request

In [ ]:
records = []

In [ ]:
# fetch recent observations per region
for region in target_regions:
    region_records = get_species_observations(api_key, species_code, region, back=30)
    if region_records:
        for record in region_records:
            record['region_code'] = region
        records.extend(region_records)
        print(f'Added {region} to records.')
    else:
        print(f'No records found for {region}.')
print(f'Fetched {len(records)} total records across {len(target_regions)} regions.')

#### Raw JSON Inspection

In [ ]:
# inspect single JSON record
records[0]

#### Schema & Field Overview

In [ ]:
ebird_df = pd.DataFrame(records)

In [ ]:
# preview top 5 rows
ebird_df.head()

In [ ]:
# display columns
ebird_df.columns

In [ ]:
# check data types
ebird_df.dtypes

#### Data Quality

In [ ]:
# row & column count
ebird_df.shape

In [ ]:
# basic stats
ebird_df.describe()

In [ ]:
# null count per column 
ebird_df.isnull().sum()

In [ ]:
# check for any duplicate records
print(f'Duplicate Records: {ebird_df.duplicated().sum()}')

In [ ]:
# coordinate coverage
print(f'missing lat/lng: \n{ebird_df[['lat', 'lng']].isnull().mean()}')

#### Geographic Coverage

In [ ]:
# total sightings per region
ebird_df['region_code'].value_counts()

In [ ]:
# hotspot vs personal location breakdown
ebird_df['locationPrivate'].value_counts()

#### Observation Patterns

In [ ]:
# unique locations observed
ebird_df['locId'].nunique()

In [ ]:
# unique observers
ebird_df['subId'].nunique()

In [ ]:
# observation count distribution
ebird_df['howMany'].describe()

In [ ]:
ebird_df['howMany'].value_counts().head()

In [ ]:
# null count for 'howMany' (affects flock size analysis)
print(f"Missing 'howMany': {ebird_df['howMany'].isnull().sum()}")

#### Temporal Coverage

In [ ]:
# convert to datetime
ebird_df['obsDt'] = pd.to_datetime(ebird_df['obsDt'], format='mixed')

In [ ]:
# date range
print(ebird_df['obsDt'].min(), 'to', ebird_df['obsDt'].max())

In [ ]:
print('missing timestamps: ', ebird_df['obsDt'].isnull().sum()) 

#### Key Findings

**Volume**
- **Finding:** 8,969 total sightings across 5 US states over the last 30 days.
- **Note:** eBird API returns only the last 30 days, complementary to GBIF (historical) rather than overlapping.

**Schema**
- **Finding:** 14 columns total, clean and well-structured JSON with no nulls except `howMany` (339 missing, 3.8%).
- **Note:** `obsDt` stored as string and will be cast to datetime in Silver layer. All other types are correct.

**Data Quality**
- **Finding:** No duplicate records. 100% coordinate coverage (lat/lng fully populated).
- **Note:** `howMany` nulls likely represent presence-only observations where count was not recorded. Will handle with `fillna` or filter in Silver layer.

**Geographic Coverage**
- **Finding:** Texas dominates with 5,090 sightings (56.7%), followed by Arizona (18%), California (16.8%), New Mexico (5.2%), and Oklahoma (3.3%).
- **Note:** Distribution is consistent with the species' known range. Texas is the core habitat.

**Coordinates**
- **Finding:** Latitude ranges from 25.9°N to 42.0°N, longitude from -123.3°W to -93.7°W, covering the southwestern US as expected.
- **Note:** Mean coordinates (31.8°N, -104.2°W) center around west Texas/New Mexico, consistent with the species' core range.

**Location Privacy**
- **Finding:** 62.8% of sightings are from private locations, 37.2% from eBird hotspots.
- **Note:** Private locations have exact coordinates with no obscuring, unlike some other biodiversity datasets.

**Observation Patterns**
- **Finding:** Mean flock size is 9.8 birds, but median is 3, distribution is right-skewed driven by a max of 8,000. Most sightings (2,394) report just 1 bird.
- **Note:** Large flock outliers (max 8,000) are valid for this species. Great-tailed Grackles are known to form massive urban roosts. Will keep outliers in Bronze and flag in Silver.

**Temporal Coverage**
- **Finding:** Records span March 10 to April 8, 2026, a full 30-day window with no missing timestamps.
- **Note:** Pipeline will run on a rolling 30-day basis to keep data current.